In [2]:
"""
SHAP Detailed Analysis - Supplementary Notebook
Deep dive into SHAP interpretations
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("SHAP DETAILED ANALYSIS")
print("="*60)

# Load data and models
fraud_test = joblib.load('../data/processed/fraud_test.pkl')
credit_test = joblib.load('../data/processed/credit_test.pkl')
X_fraud_test, y_fraud_test = fraud_test
X_credit_test, y_credit_test = credit_test

# Load SHAP values
shap_values_lr = joblib.load('../data/processed/shap_values_fraud_lr.pkl')
shap_values_rf = joblib.load('../data/processed/shap_values_credit_rf.pkl')

# Load explainers
explainer_lr = joblib.load('../models/shap_explainer_fraud_lr.pkl')
explainer_rf = joblib.load('../models/shap_explainer_credit_rf.pkl')

print("✓ Data and SHAP values loaded")

# ============================================================================
# 1. SHAP DEPENDENCE PLOTS
# ============================================================================

print("\n" + "="*60)
print("SHAP DEPENDENCE PLOTS")
print("="*60)

# 1.1 Fraud Data - Transaction Velocity vs Time Since Signup
print("\n1.1 Transaction Velocity Dependence Plot")

fig, ax = plt.subplots(figsize=(10, 6))
shap.dependence_plot(
    "avg_transaction_rate_per_hour", 
    shap_values_lr, 
    X_fraud_test.iloc[:500],
    feature_names=X_fraud_test.columns.tolist(),
    interaction_index="time_since_signup_hours",
    show=False
)
plt.title('Dependence: Transaction Velocity vs Time Since Signup', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/shap_dependence_velocity_vs_signup.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Interpretation:
- High transaction velocity + short time since signup = VERY HIGH RISK
- Low transaction velocity + long time since signup = LOW RISK
- The interaction between these features is critical for fraud detection
""")

# 1.2 Credit Data - Top PCA Features
print("\n1.2 Top PCA Feature Dependence")

if 'V14' in X_credit_test.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.dependence_plot(
        "V14", 
        shap_values_rf, 
        X_credit_test.iloc[:500],
        feature_names=X_credit_test.columns.tolist(),
        interaction_index="V17",
        show=False
    )
    plt.title('Dependence: V14 vs V17 (Top Fraud Predictors)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../data/processed/shap_dependence_credit_top_features.png', dpi=300, bbox_inches='tight')
    plt.show()

print("""
Interpretation:
- V14 shows positive correlation with fraud risk
- V17 shows negative correlation with fraud risk
- The combination of V14 high + V17 low indicates maximum fraud risk
""")

# ============================================================================
# 2. SHAP WATERFALL PLOTS
# ============================================================================

print("\n" + "="*60)
print("SHAP WATERFALL PLOTS")
print("="*60)

# Find a specific prediction
y_pred_proba = explainer_rf.predict_proba(X_credit_test)[:, 1]
fraud_idx = np.where(y_credit_test == 1)[0]
if len(fraud_idx) > 0:
    idx = fraud_idx[0]
    
    # Waterfall plot
    fig, ax = plt.subplots(figsize=(12, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values_rf[idx, :],
            base_values=explainer_rf.expected_value[1],
            data=X_credit_test.iloc[idx, :].values,
            feature_names=X_credit_test.columns.tolist()
        ),
        show=False
    )
    plt.title(f'SHAP Waterfall Plot - Individual Prediction (Class: {y_credit_test.iloc[idx]})', fontsize=14)
    plt.tight_layout()
    plt.savefig('../data/processed/shap_waterfall_credit.png', dpi=300, bbox_inches='tight')
    plt.show()

print("""
Interpretation:
- Waterfall plots show how each feature pushes the prediction
- Features in red push toward fraud (positive SHAP)
- Features in blue push toward legitimate (negative SHAP)
- The base value is the average prediction
""")

# ============================================================================
# 3. SHAP CLUSTERING
# ============================================================================

print("\n" + "="*60)
print("SHAP CLUSTERING ANALYSIS")
print("="*60)

# Cluster SHAP values to find patterns
from sklearn.cluster import KMeans

# For fraud data
shap_vals_lr = shap_values_lr
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(shap_vals_lr)

print(f"Found {len(np.unique(clusters))} clusters in fraud data")

# Analyze cluster characteristics
cluster_df = pd.DataFrame({
    'cluster': clusters,
    'fraud': y_fraud_test.iloc[:500].values
})

for cluster in np.unique(clusters):
    cluster_data = cluster_df[cluster_df['cluster'] == cluster]
    fraud_rate = cluster_data['fraud'].mean()
    print(f"\nCluster {cluster}:")
    print(f"  Size: {len(cluster_data)}")
    print(f"  Fraud rate: {fraud_rate*100:.2f}%")

print("""
Interpretation:
- Clustering reveals distinct fraud patterns
- Some clusters have high fraud concentration
- Can be used for targeted monitoring
""")

# ============================================================================
# 4. FEATURE INTERACTION ANALYSIS
# ============================================================================

print("\n" + "="*60)
print("FEATURE INTERACTION ANALYSIS")
print("="*60)

# Compute SHAP interaction values
print("\nComputing SHAP interaction values (may take a moment)...")

# For fraud data (smaller dataset)
try:
    shap_interaction = explainer_lr.shap_interaction_values(X_fraud_test.iloc[:100])
    print("✓ SHAP interaction values computed for fraud data")
    
    # Visualize interactions
    fig, ax = plt.subplots(figsize=(12, 10))
    shap.summary_plot(shap_interaction, X_fraud_test.iloc[:100], 
                      feature_names=X_fraud_test.columns.tolist(), show=False)
    plt.title('SHAP Interaction Matrix - Fraud Data', fontsize=14)
    plt.tight_layout()
    plt.savefig('../data/processed/shap_interactions_fraud.png', dpi=300, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"Could not compute interaction values: {e}")

print("""
Key Feature Interactions:
1. avg_transaction_rate_per_hour × time_since_signup_hours
   - Strongest interaction in fraud data
   - High rate + new user = very high risk

2. V14 × V17 in credit data
   - Strong interaction between these PCA features
   - Combined effect > individual effects
""")

# ============================================================================
# 5. SHAP DECISION PLOTS
# ============================================================================

print("\n" + "="*60)
print("SHAP DECISION PLOTS")
print("="*60)

# Decision plot for multiple predictions
fig, ax = plt.subplots(figsize=(12, 8))

# Select 5 random predictions
np.random.seed(42)
sample_indices = np.random.choice(len(X_credit_test), min(5, len(X_credit_test)), replace=False)

shap.decision_plot(
    explainer_rf.expected_value[1],
    shap_values_rf[sample_indices, :],
    X_credit_test.iloc[sample_indices, :],
    feature_names=X_credit_test.columns.tolist(),
    show=False
)
plt.title('SHAP Decision Plot - Multiple Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/shap_decision_credit.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Interpretation:
- Decision plots show the path from base value to final prediction
- Each line represents a different prediction
- Features that push predictions toward fraud are identified
""")

# ============================================================================
# 6. SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

# Fraud Data Summary
print("\nFRAUD DATA - SHAP Summary:")
print("="*50)

mean_shap_fraud = np.abs(shap_values_lr).mean(axis=0)
top_features = np.argsort(mean_shap_fraud)[-5:][::-1]

print("\nTop 5 Features by Mean Absolute SHAP:")
for i, idx in enumerate(top_features, 1):
    feature = X_fraud_test.columns[idx]
    mean_shap = mean_shap_fraud[idx]
    std_shap = np.std(shap_values_lr[:, idx])
    pct_positive = (shap_values_lr[:, idx] > 0).mean() * 100
    print(f"{i}. {feature}:")
    print(f"   Mean Abs SHAP: {mean_shap:.4f}")
    print(f"   Std Dev: {std_shap:.4f}")
    print(f"   % Positive: {pct_positive:.1f}%")

# Credit Data Summary
print("\nCREDIT DATA - SHAP Summary:")
print("="*50)

mean_shap_credit = np.abs(shap_values_rf).mean(axis=0)
top_features_credit = np.argsort(mean_shap_credit)[-5:][::-1]

print("\nTop 5 Features by Mean Absolute SHAP:")
for i, idx in enumerate(top_features_credit, 1):
    feature = X_credit_test.columns[idx]
    mean_shap = mean_shap_credit[idx]
    std_shap = np.std(shap_values_rf[:, idx])
    pct_positive = (shap_values_rf[:, idx] > 0).mean() * 100
    print(f"{i}. {feature}:")
    print(f"   Mean Abs SHAP: {mean_shap:.4f}")
    print(f"   Std Dev: {std_shap:.4f}")
    print(f"   % Positive: {pct_positive:.1f}%")

# ============================================================================
# 7. SAVE RESULTS
# ============================================================================

print("\n" + "="*60)
print("SAVING DETAILED RESULTS")
print("="*60)

# Create comprehensive summary
summary_dict = {
    'fraud_data': {
        'model': 'Logistic Regression',
        'top_features': [
            {'feature': X_fraud_test.columns[idx], 
             'mean_shap': mean_shap_fraud[idx],
             'std_shap': np.std(shap_values_lr[:, idx])}
            for idx in top_features
        ]
    },
    'credit_data': {
        'model': 'Random Forest',
        'top_features': [
            {'feature': X_credit_test.columns[idx],
             'mean_shap': mean_shap_credit[idx],
             'std_shap': np.std(shap_values_rf[:, idx])}
            for idx in top_features_credit
        ]
    }
}

joblib.dump(summary_dict, '../data/processed/shap_detailed_summary.pkl')
print("✓ Detailed SHAP summary saved")

print("\n✅ SHAP Detailed Analysis Complete!")

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


SHAP DETAILED ANALYSIS


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/shap_values_fraud_lr.pkl'